# Early Exit Adapter Pipeline

Colab-friendly notebook wrapper around the `early_exit_adapters` package.

## 1. Clone repo and enter the Colab working folder

Set `REPO_URL` to the GitHub repo that contains this package. The cell clones into `/content/Speculative_decoding`, updates it if it already exists, changes the process working directory, and verifies the package is present.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/YOUR_GITHUB_USER/Speculative_decoding.git"  # <-- change this
REPO_DIR = Path("/content/Speculative_decoding")

if "YOUR_GITHUB_USER" in REPO_URL:
    raise ValueError("Set REPO_URL to your actual GitHub repo URL before running this cell.")

if REPO_DIR.exists():
    print(f"Repo already exists at {REPO_DIR}; pulling latest changes...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())
print("top-level files:", sorted(p.name for p in Path.cwd().iterdir())[:20])

assert Path("early_exit_adapters").is_dir(), "Missing early_exit_adapters package. Are you in the cloned repo root?"
assert Path("notebooks/run_full_pipeline.ipynb").exists(), "Notebook file not found in expected location."
print("Colab folder check passed.")

## 2. Install/import dependencies

In [ ]:
%pip install -U torch transformers datasets pandas matplotlib tqdm wandb python-dotenv huggingface_hub accelerate safetensors optuna

import pandas as pd

from early_exit_adapters.base_speculative_decode import (
    load_truncated_qwen_draft_model,
    run_speculative_eval,
)
from early_exit_adapters.data import load_fineweb_stream
from early_exit_adapters.evaluate import layers_kl_over_data_rows
from early_exit_adapters.hf_utils import load_adapters_from_hf, upload_adapter_folder_to_hf
from early_exit_adapters.model import load_lm_model_and_tokenizer
from early_exit_adapters.pipeline import main
from early_exit_adapters.train import train_initial_adapters
from early_exit_adapters.visualization import (
    plot_baseline_all_metrics,
    plot_baseline_metric_by_layer,
    plot_baseline_normalized_metrics,
    plot_baseline_vs_adapted,
)
from opt_sd.scripts.sd_tree import speculative_decode_tree


## 3. Mount Drive and choose where useful artifacts are saved

Checkpoints, final adapters, logs, metadata, and evaluation JSON/JSONL should go somewhere persistent. In Colab, `/content` disappears when the runtime ends; Google Drive survives.

In [ ]:
from pathlib import Path
import os

SAVE_TO_DRIVE = True
RUN_NAME = "qwen35_2b_residual_adapters"

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/early_exit_adapters")
else:
    ARTIFACT_ROOT = Path("/content/artifacts/early_exit_adapters")

OUT_DIR = ARTIFACT_ROOT / RUN_NAME
EVAL_DIR = ARTIFACT_ROOT / f"{RUN_NAME}_eval"
BASELINE_EVAL_DIR = EVAL_DIR / "baseline"
ADAPTED_EVAL_DIR = EVAL_DIR / "adapted"

for path in [OUT_DIR, BASELINE_EVAL_DIR, ADAPTED_EVAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Saving training artifacts to:", OUT_DIR)
print("Saving eval artifacts to:", EVAL_DIR)

## 4. Load `.env`

In Colab you can either upload a `.env` file to the repo root, store one in Drive, or set secrets directly in `os.environ` before logging into W&B/Hugging Face.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # loads .env from the cloned repo root if present

# Optional Colab-only fallback, if you store .env next to the artifacts in Drive:
# load_dotenv(ARTIFACT_ROOT / ".env")

## 5. Load model/tokenizer

In [ ]:
model_name = "Qwen/Qwen3.5-2B"

model, tokenizer, device = load_lm_model_and_tokenizer(model_name=model_name)
device

## 6. Load FineWeb-Edu stream

In [ ]:
dataset = load_fineweb_stream(
    dataset_name="HuggingFaceFW/fineweb-edu",
    split="train",
    seed=42,
    buffer_size=10_000,
)

## 7. Train adapters

In [ ]:
adapters, logs_df = train_initial_adapters(
    model=model,
    tokenizer=tokenizer,
    dataset=dataset,
    candidate_layers=[4, 8, 12, 16, 18, 20],
    max_steps=500,
    seq_len=128,
    lr=1e-4,
    temperature=2.0,
    top_k=5,
    log_every=20,
    eval_step=100,
    eval_size=32,
    save_every=100,
    out_dir=str(OUT_DIR),
    use_wandb=True,
    wandb_project="qwen35-early-exit-adapters",
    wandb_run_name=RUN_NAME,
    wandb_mode=None,
)


## 8. Confirm useful files were saved

In [ ]:
logs_df.tail()

In [ ]:
for path in sorted(OUT_DIR.glob("*")):
    print(path.name, path.stat().st_size, "bytes")

assert (OUT_DIR / "adapters_final.pt").exists(), "Final adapter checkpoint was not saved."
assert (OUT_DIR / "training_logs.jsonl").exists(), "Training logs were not saved."
print("Training artifact check passed:", OUT_DIR)

## 9. Run baseline/adapted layer evaluation

In [ ]:
baseline_dataset = load_fineweb_stream(
    dataset_name="HuggingFaceFW/fineweb-edu",
    split="train",
    seed=42,
    buffer_size=10_000,
)

baseline_records_df, baseline_summary = layers_kl_over_data_rows(
    model=model,
    tokenizer=tokenizer,
    dataset=baseline_dataset,
    candidate_layers=[4, 8, 12, 16, 18, 20, 22, 24],
    seq_len=128,
    num_eval_rows=100,
    temperature=2.0,
    top_k=5,
    out_dir=str(BASELINE_EVAL_DIR),
)

adapted_dataset = load_fineweb_stream(
    dataset_name="HuggingFaceFW/fineweb-edu",
    split="train",
    seed=999,
    buffer_size=10_000,
)

adapted_records_df, adapted_summary = layers_kl_over_data_rows(
    model=model,
    tokenizer=tokenizer,
    dataset=adapted_dataset,
    candidate_layers=[4, 8, 12, 16, 18, 20],
    adapters=adapters,
    run_name="adapted",
    seq_len=128,
    num_eval_rows=100,
    temperature=2.0,
    top_k=5,
    out_dir=str(ADAPTED_EVAL_DIR),
)

compare_summary = pd.concat([baseline_summary, adapted_summary], ignore_index=True)

## 10. Plot baseline metrics

In [ ]:
plot_baseline_metric_by_layer(
    baseline_summary,
    metric="kl_to_teacher",
    title="Baseline KL to final model by layer",
)
plot_baseline_all_metrics(baseline_summary)
plot_baseline_normalized_metrics(baseline_summary)

## 11. Plot baseline vs adapted metrics

In [ ]:
plot_baseline_vs_adapted(compare_summary, metric="kl_to_teacher")
plot_baseline_vs_adapted(compare_summary, metric="ce")
plot_baseline_vs_adapted(compare_summary, metric="top1_teacher_agreement")
plot_baseline_vs_adapted(compare_summary, metric="topk_overlap")
plot_baseline_vs_adapted(compare_summary, metric="accept_proxy_exact")
plot_baseline_vs_adapted(compare_summary, metric="accept_proxy_sampled")

## 12. Run speculative decoding benchmark

This compares baseline early-layer speculative decoding with adapted early-layer speculative decoding. The adapted run uses the adapter from the same layer being evaluated.

In [ ]:
spec_dataset = load_fineweb_stream(
    dataset_name="HuggingFaceFW/fineweb-edu",
    split="train",
    seed=123,
    buffer_size=10_000,
)

baseline_layer_index = 8

baseline_draft_model = load_truncated_qwen_draft_model(
    model_name=model_name,
    layer_index=baseline_layer_index,
    device=device,
)

baseline_spec_result = run_speculative_eval(
    model=model,
    draft_model=baseline_draft_model,
    tokenizer=tokenizer,
    ds=spec_dataset,
    layer_index=baseline_layer_index,
    gamma=3,
    seqlen=10,
    num_prefix_tokens=10,
    num_of_examples=100,
    adapter=None,
    draft_temperature=0.8,
    device_type=device,
)

del baseline_draft_model
import gc
import torch
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

baseline_spec_result


In [ ]:
spec_dataset = load_fineweb_stream(
    dataset_name="HuggingFaceFW/fineweb-edu",
    split="train",
    seed=123,
    buffer_size=10_000,
)

adapted_layer_index = 12

adapted_draft_model = load_truncated_qwen_draft_model(
    model_name=model_name,
    layer_index=adapted_layer_index,
    device=device,
)

adapted_spec_result = run_speculative_eval(
    model=model,
    draft_model=adapted_draft_model,
    tokenizer=tokenizer,
    ds=spec_dataset,
    layer_index=adapted_layer_index,
    gamma=4,
    seqlen=10,
    num_prefix_tokens=10,
    num_of_examples=100,
    adapter=adapters[str(adapted_layer_index)],
    draft_temperature=0.8,
    device_type=device,
)

del adapted_draft_model
import gc
import torch
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

adapted_spec_result


In [ ]:
spec_results_df = pd.DataFrame([
    {"run_name": "baseline", **baseline_spec_result},
    {"run_name": "adapted", **adapted_spec_result},
])

spec_results_path = EVAL_DIR / "speculative_decode_results.jsonl"
spec_results_df.to_json(spec_results_path, orient="records", lines=True)
print("saved:", spec_results_path)
spec_results_df

## 13. Upload adapters to Hugging Face

In [ ]:
# upload_adapter_folder_to_hf(
#     repo_id="Maorb23/qwen35-2b-early-exit-adapters",
#     folder_path=str(OUT_DIR),
# )

## 14. Reload adapters from Hugging Face for sanity check

In [ ]:
# reloaded_adapters = load_adapters_from_hf(
#     model=model,
#     repo_id="Maorb23/qwen35-2b-early-exit-adapters",
#     filename="adapters_final.pt",
#     device=device,
# )
# reloaded_adapters

## 15. Optional quick smoke test cell

Use this before a long run if you want to verify the whole pipeline path with tiny settings.

In [ ]:
# smoke_adapters, smoke_logs_df = main(
#     model_name="Qwen/Qwen3.5-2B",
#     candidate_layers=[4, 8],
#     max_steps=5,
#     seq_len=128,
#     eval_size=2,
#     eval_step=5,
#     save_every=5,
#     out_dir=str(ARTIFACT_ROOT / "smoke_test"),
#     use_wandb=True,
#     wandb_project="qwen35-early-exit-adapters",
#     wandb_run_name="smoke_test",
#     wandb_mode="offline",
#     upload_to_hf=False,
# )
# smoke_logs_df.tail()

## 16. OPT-tree speculative decoding vs target-only baseline

This benchmark uses a normal small draft model plus the already-loaded Qwen3.5-2B target model. It does not use the early-exit adapters. The OPT-tree code first tries 4D tree attention for verification and falls back to batched path verification if the model/backend rejects the custom mask.

In [ ]:
import gc
import json
import os
import time

import torch
from transformers import AutoModelForCausalLM

TREE_TARGET_MODEL_NAME = model_name
TREE_DRAFT_MODEL_NAME = "Qwen/Qwen3.5-0.8B"
TREE_INPUT_TEXT = "The future of artificial intelligence is"
TREE_MAX_NEW_TOKENS = 50
TREE_NODE_BUDGET = 8
TREE_MAX_DEPTH = 4
TREE_BRANCH_TOP_K = 4
TREE_DRAFT_TEMPERATURE = 1.0
TREE_USE_DRAFT_CACHE = True
TREE_LOG_TREE_NODES = False
TREE_DECODE_STEP_TEXT = False
TREE_DO_SAMPLE = False
TREE_TEMPERATURE = 0.0

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")

tree_tokenizer = tokenizer

if tree_tokenizer.pad_token_id is None:
    tree_tokenizer.pad_token_id = tree_tokenizer.eos_token_id

print("target:", TREE_TARGET_MODEL_NAME)
print("draft:", TREE_DRAFT_MODEL_NAME)
print("device:", device)

In [ ]:
tree_draft_model = AutoModelForCausalLM.from_pretrained(
    TREE_DRAFT_MODEL_NAME,
    torch_dtype=torch.bfloat16 if str(device).startswith("cuda") else torch.float32,
    token=hf_token,
    trust_remote_code=True,
).to(device)

tree_draft_model.eval()
for p in tree_draft_model.parameters():
    p.requires_grad = False

if tree_draft_model.config.vocab_size != model.config.vocab_size:
    raise ValueError(
        f"Draft/target vocab mismatch: {tree_draft_model.config.vocab_size} vs {model.config.vocab_size}. "
        "Pick a draft model with the same tokenizer/vocab family."
    )

print("loaded tree draft model")
print("draft vocab:", tree_draft_model.config.vocab_size)
print("target vocab:", model.config.vocab_size)

In [ ]:
def sync_for_timing():
    if str(device).startswith("cuda") and torch.cuda.is_available():
        torch.cuda.synchronize()


def timed_target_generate(input_text, max_new_tokens):
    inputs = tree_tokenizer(input_text, return_tensors="pt").to(device)

    sync_for_timing()
    start = time.perf_counter()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=TREE_DO_SAMPLE,
            temperature=None if not TREE_DO_SAMPLE else TREE_TEMPERATURE,
            pad_token_id=tree_tokenizer.eos_token_id,
        )

    sync_for_timing()
    elapsed = time.perf_counter() - start

    generated_tokens = int(output_ids.shape[-1] - inputs["input_ids"].shape[-1])
    return {
        "method": "target_generate",
        "elapsed_sec": elapsed,
        "generated_tokens": generated_tokens,
        "tokens_per_sec": generated_tokens / elapsed if elapsed > 0 else None,
        "text": tree_tokenizer.decode(output_ids[0], skip_special_tokens=True),
    }

baseline_tree_compare = timed_target_generate(
    input_text=TREE_INPUT_TEXT,
    max_new_tokens=TREE_MAX_NEW_TOKENS,
)

baseline_tree_compare

In [ ]:
tree_report_path = EVAL_DIR / "opt_tree_qwen35_2b_report.json"

sync_for_timing()
tree_start = time.perf_counter()

final_text, tree_all_ids, tree_step_infos, tree_report = speculative_decode_tree(
    input_text=TREE_INPUT_TEXT,
    tokenizer=tree_tokenizer,
    small_model=tree_draft_model,
    big_model=model,
    seqlen=TREE_MAX_NEW_TOKENS,
    device_type=device,
    do_sample=TREE_DO_SAMPLE,
    temperature=TREE_TEMPERATURE,
    print_logs=False,
    save_logs_path=str(tree_report_path),
    node_budget=TREE_NODE_BUDGET,
    max_depth=TREE_MAX_DEPTH,
    branch_top_k=TREE_BRANCH_TOP_K,
    draft_temperature=TREE_DRAFT_TEMPERATURE,
    prefer_tree_attention=True,
    use_draft_cache=TREE_USE_DRAFT_CACHE,
    stop_on_eos=True,
    log_tree_nodes=TREE_LOG_TREE_NODES,
    decode_step_text=TREE_DECODE_STEP_TEXT,
)

sync_for_timing()
tree_elapsed = time.perf_counter() - tree_start

prompt_ids = tree_tokenizer(TREE_INPUT_TEXT, return_tensors="pt")["input_ids"]
tree_generated_tokens = int(tree_all_ids.numel() - prompt_ids.shape[-1])

tree_compare = {
    "method": "opt_tree_speculative",
    "elapsed_sec": tree_elapsed,
    "generated_tokens": tree_generated_tokens,
    "tokens_per_sec": tree_generated_tokens / tree_elapsed if tree_elapsed > 0 else None,
    "mean_tree_nodes": tree_report["stats"].get("mean_tree_nodes"),
    "mean_tree_depth": tree_report["stats"].get("mean_tree_depth"),
    "mean_accepted_draft_tokens": tree_report["stats"].get("mean_accepted_draft_tokens"),
    "mean_draft_forward_passes": tree_report["stats"].get("mean_draft_forward_passes"),
    "mean_target_forward_passes": tree_report["stats"].get("mean_target_forward_passes"),
    "draft_cache_used": sorted({
        step["tree"].get("draft_cache_used")
        for step in tree_report.get("steps", [])
    }),
    "target_forward_methods": sorted({
        step["target_forward"].get("target_forward_method")
        for step in tree_report.get("steps", [])
    }),
    "text": final_text,
}

tree_compare

In [ ]:
opt_tree_comparison_df = pd.DataFrame([
    baseline_tree_compare,
    tree_compare,
])

opt_tree_comparison_path = EVAL_DIR / "opt_tree_vs_baseline_comparison.jsonl"
opt_tree_comparison_df.to_json(opt_tree_comparison_path, orient="records", lines=True)

print("saved tree report:", tree_report_path)
print("saved comparison:", opt_tree_comparison_path)
opt_tree_comparison_df[[
    "method",
    "elapsed_sec",
    "generated_tokens",
    "tokens_per_sec",
    "mean_tree_nodes",
    "mean_tree_depth",
    "mean_accepted_draft_tokens",
    "mean_draft_forward_passes",
    "mean_target_forward_passes",
    "draft_cache_used",
    "target_forward_methods",
]]


In [ ]:
del tree_draft_model

gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("released tree draft model")

## 17. Optuna search for faster OPT-tree settings

This searches small OPT-tree configurations and compares them by tokens/sec. Keep the first run small because every trial is a real generation benchmark.

In [ ]:
import gc
import json
import time

import optuna
import torch
from transformers import AutoModelForCausalLM

OPTUNA_N_TRIALS = 20
OPTUNA_TIMEOUT_SEC = None  # Example: 900 for a 15 minute cap.
OPTUNA_MAX_NEW_TOKENS = 50
OPTUNA_INPUT_TEXT = TREE_INPUT_TEXT if "TREE_INPUT_TEXT" in globals() else "The future of artificial intelligence is"
OPTUNA_DRAFT_MODEL_NAME = "Qwen/Qwen3.5-0.8B"
OPTUNA_STUDY_NAME = "opt_tree_qwen35_2b_speed"
OPTUNA_STORAGE_PATH = EVAL_DIR / "opt_tree_optuna.sqlite3"
OPTUNA_RESULTS_PATH = EVAL_DIR / "opt_tree_optuna_trials.csv"

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
tree_tokenizer = tokenizer

if tree_tokenizer.pad_token_id is None:
    tree_tokenizer.pad_token_id = tree_tokenizer.eos_token_id

if "tree_draft_model_for_optuna" not in globals():
    tree_draft_model_for_optuna = AutoModelForCausalLM.from_pretrained(
        OPTUNA_DRAFT_MODEL_NAME,
        torch_dtype=torch.bfloat16 if str(device).startswith("cuda") else torch.float32,
        token=hf_token,
        trust_remote_code=True,
    ).to(device)
    tree_draft_model_for_optuna.eval()
    for p in tree_draft_model_for_optuna.parameters():
        p.requires_grad = False

print("Optuna draft:", OPTUNA_DRAFT_MODEL_NAME)
print("study storage:", OPTUNA_STORAGE_PATH)

In [ ]:
def optuna_sync_for_timing():
    if str(device).startswith("cuda") and torch.cuda.is_available():
        torch.cuda.synchronize()


def measure_target_baseline_for_optuna():
    inputs = tree_tokenizer(OPTUNA_INPUT_TEXT, return_tensors="pt").to(device)
    optuna_sync_for_timing()
    start = time.perf_counter()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=OPTUNA_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tree_tokenizer.eos_token_id,
        )

    optuna_sync_for_timing()
    elapsed = time.perf_counter() - start
    generated_tokens = int(output_ids.shape[-1] - inputs["input_ids"].shape[-1])
    return {
        "elapsed_sec": elapsed,
        "generated_tokens": generated_tokens,
        "tokens_per_sec": generated_tokens / elapsed if elapsed > 0 else 0.0,
    }

optuna_baseline = measure_target_baseline_for_optuna()
optuna_baseline

In [ ]:
def opt_tree_speed_objective(trial):
    node_budget = trial.suggest_categorical("node_budget", [2, 4, 8, 12, 16])
    max_depth = trial.suggest_int("max_depth", 2, 6)
    branch_top_k = trial.suggest_int("branch_top_k", 1, min(8, node_budget))
    draft_temperature = trial.suggest_float("draft_temperature", 0.5, 1.3)
    threshold = trial.suggest_float("threshold", 0.0, 0.05)

    # Avoid obviously wasteful shapes where the tree is very wide but cannot go deep.
    if branch_top_k > node_budget:
        raise optuna.TrialPruned()

    optuna_sync_for_timing()
    start = time.perf_counter()

    final_text, tree_all_ids, tree_step_infos, tree_report = speculative_decode_tree(
        input_text=OPTUNA_INPUT_TEXT,
        tokenizer=tree_tokenizer,
        small_model=tree_draft_model_for_optuna,
        big_model=model,
        seqlen=OPTUNA_MAX_NEW_TOKENS,
        device_type=device,
        do_sample=False,
        temperature=0.0,
        print_logs=False,
        save_logs_path=None,
        node_budget=node_budget,
        max_depth=max_depth,
        threshold=threshold,
        branch_top_k=branch_top_k,
        draft_temperature=draft_temperature,
        prefer_tree_attention=True,
        use_draft_cache=True,
        stop_on_eos=False,
        log_tree_nodes=False,
        decode_step_text=False,
    )

    optuna_sync_for_timing()
    elapsed = time.perf_counter() - start

    prompt_ids = tree_tokenizer(OPTUNA_INPUT_TEXT, return_tensors="pt")["input_ids"]
    generated_tokens = int(tree_all_ids.numel() - prompt_ids.shape[-1])
    tokens_per_sec = generated_tokens / elapsed if elapsed > 0 else 0.0

    stats = tree_report["stats"]
    target_forward_methods = sorted({
        step["target_forward"].get("target_forward_method")
        for step in tree_report.get("steps", [])
    })
    draft_cache_used = sorted({
        step["tree"].get("draft_cache_used")
        for step in tree_report.get("steps", [])
    })

    trial.set_user_attr("elapsed_sec", elapsed)
    trial.set_user_attr("generated_tokens", generated_tokens)
    trial.set_user_attr("tokens_per_sec", tokens_per_sec)
    trial.set_user_attr("baseline_tokens_per_sec", optuna_baseline["tokens_per_sec"])
    trial.set_user_attr("speedup_vs_baseline", tokens_per_sec / optuna_baseline["tokens_per_sec"])
    trial.set_user_attr("mean_tree_nodes", stats.get("mean_tree_nodes"))
    trial.set_user_attr("mean_tree_depth", stats.get("mean_tree_depth"))
    trial.set_user_attr("mean_accepted_draft_tokens", stats.get("mean_accepted_draft_tokens"))
    trial.set_user_attr("mean_draft_forward_passes", stats.get("mean_draft_forward_passes"))
    trial.set_user_attr("mean_target_forward_passes", stats.get("mean_target_forward_passes"))
    trial.set_user_attr("target_forward_methods", target_forward_methods)
    trial.set_user_attr("draft_cache_used", draft_cache_used)

    return tokens_per_sec

study = optuna.create_study(
    study_name=OPTUNA_STUDY_NAME,
    storage=f"sqlite:///{OPTUNA_STORAGE_PATH}",
    direction="maximize",
    load_if_exists=True,
)

study.optimize(
    opt_tree_speed_objective,
    n_trials=OPTUNA_N_TRIALS,
    timeout=OPTUNA_TIMEOUT_SEC,
    gc_after_trial=True,
    show_progress_bar=True,
)

print("best value tokens/sec:", study.best_value)
print("best params:", study.best_params)
print("best attrs:", study.best_trial.user_attrs)

In [ ]:
optuna_trials_df = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state"))
optuna_trials_df.to_csv(OPTUNA_RESULTS_PATH, index=False)

print("saved study:", OPTUNA_STORAGE_PATH)
print("saved trials:", OPTUNA_RESULTS_PATH)

cols = [
    "number",
    "value",
    "params_node_budget",
    "params_max_depth",
    "params_branch_top_k",
    "params_draft_temperature",
    "params_threshold",
    "user_attrs_elapsed_sec",
    "user_attrs_speedup_vs_baseline",
    "user_attrs_mean_accepted_draft_tokens",
    "user_attrs_mean_draft_forward_passes",
    "user_attrs_target_forward_methods",
    "state",
]

available_cols = [c for c in cols if c in optuna_trials_df.columns]
optuna_trials_df.sort_values("value", ascending=False)[available_cols].head(10)

In [ ]:
# Optional cleanup when you are done tuning.
# del tree_draft_model_for_optuna
# gc.collect()
# torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 18. vLLM speculative decoding benchmark

This is the first vLLM speed benchmark: target-only generation versus vLLM built-in `draft_model` speculative decoding with `Qwen/Qwen3.5-0.8B` as the draft for `Qwen/Qwen3.5-2B`. Run it in a CUDA/Linux, WSL, Kaggle, or cloud GPU environment where vLLM is installed.

In [ ]:
# vLLM is typically installed in the GPU runtime, not the local Windows notebook host.
# Uncomment in a fresh CUDA/Linux environment if needed:
# %pip install -U vllm

import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("vllm") is None:
    raise RuntimeError("vLLM is not installed in this kernel. Install vLLM in a CUDA/Linux GPU environment, then rerun this cell.")

repo_root = Path.cwd()
if not (repo_root / "early_exit_adapters").exists() and (repo_root.parent / "early_exit_adapters").exists():
    repo_root = repo_root.parent

VLLM_TARGET_MODEL = "Qwen/Qwen3.5-2B"
VLLM_DRAFT_MODEL = "Qwen/Qwen3.5-0.8B"
VLLM_NUM_SPECULATIVE_TOKENS = 4
VLLM_MAX_TOKENS = 128
VLLM_REPEATS = 3
VLLM_MAX_MODEL_LEN = 4096
VLLM_MAX_NUM_SEQS = 4
VLLM_GPU_MEMORY_UTILIZATION = 0.75
VLLM_RESULTS_PATH = (EVAL_DIR if "EVAL_DIR" in globals() else repo_root / "results") / "vllm_qwen35_2b_08b.json"
VLLM_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

if os.environ.get("HF_TOKEN") is None and os.environ.get("HUGGINGFACE_TOKEN"):
    os.environ["HF_TOKEN"] = os.environ["HUGGINGFACE_TOKEN"]

# Free HF/PyTorch models from previous notebook sections before vLLM reserves GPU memory.
for name in ["model", "tree_draft_model", "tree_draft_model_for_optuna", "draft_model"]:
    if name in globals():
        del globals()[name]
if "torch" in globals() and torch.cuda.is_available():
    torch.cuda.empty_cache()

cmd = [
    sys.executable,
    "-m",
    "early_exit_adapters.vllm_speculative_benchmark",
    "--target-model",
    VLLM_TARGET_MODEL,
    "--draft-model",
    VLLM_DRAFT_MODEL,
    "--mode",
    "both",
    "--num-speculative-tokens",
    str(VLLM_NUM_SPECULATIVE_TOKENS),
    "--max-tokens",
    str(VLLM_MAX_TOKENS),
    "--repeats",
    str(VLLM_REPEATS),
    "--max-model-len",
    str(VLLM_MAX_MODEL_LEN),
    "--max-num-seqs",
    str(VLLM_MAX_NUM_SEQS),
    "--gpu-memory-utilization",
    str(VLLM_GPU_MEMORY_UTILIZATION),
    "--out",
    str(VLLM_RESULTS_PATH),
    "--quiet",
]

completed = subprocess.run(cmd, cwd=repo_root, text=True, capture_output=True)
if completed.returncode != 0:
    print(completed.stdout)
    print(completed.stderr)
    completed.check_returncode()

vllm_results = json.loads(VLLM_RESULTS_PATH.read_text(encoding="utf-8"))
vllm_results["comparison"]
